# gpt-oss20bのSFTによる学習

# 実験結果サマリ

- r=256、epoc=30まで上げて学習させても、うまくいかず
- gpt-ossがMoEモデルだったためうまくいかないのか？→Qwen3.5 27bでリベンジ（へ続く）

### import

In [1]:
import json
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel
import torch
from trl import SFTConfig, SFTTrainer

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_236/1028993508.py:5: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### モデルのロード


In [2]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b", 
    max_seq_length = 2048,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.3.8: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260322. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [01:15<00:00, 18.76s/it]


### LoRAアダプタの設定

In [3]:

model = FastLanguageModel.get_peft_model(
    model,
    r = 256, # 5,000件ならランクを少し上げて表現力を確保
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
)

Unsloth: Detected MoE model with num_experts = 32 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
Unsloth: Making `model.base_model.model.model` require gradients


### 学習データをロード

In [4]:
data_list = []
with open("sft_data_generated.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data_list.append(json.loads(line))

dataset = Dataset.from_list(data_list)

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = [f"### 質問:\n{i}\n\n### 回答:\n{o}" for i, o in zip(instructions, outputs)]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)


Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7056/7056 [00:00<00:00, 433353.48 examples/s]


### 学習

In [5]:
sft_config = SFTConfig(
    output_dir = "kakidash_ultimate_expert",
    max_seq_length = 2048,           # ここで指定します
    per_device_train_batch_size = 16, 
    gradient_accumulation_steps = 4,
    warmup_ratio = 0.1,
    num_train_epochs = 30,            
    learning_rate = 1e-4, 
    bf16 = True,
    logging_steps = 10,
    optim = "adamw_8bit",
    lr_scheduler_type = "cosine",
    save_strategy = "epoch",
    report_to = "none",
    dataset_text_field = "text",     # 加工済みデータセットでもここで明示
)

In [6]:

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    processing_class = tokenizer,
    args = sft_config,               # TrainingArguments ではなく SFTConfig を渡す
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=24): 100%|█████████████████████████████████████████████████████████████████████████████████████| 7056/7056 [00:07<00:00, 928.13 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,056 | Num Epochs = 30 | Total steps = 3,330
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 4 x 1) = 64
 "-____-"     Trainable parameters = 127,401,984 of 21,042,159,168 (0.61% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Autotune Choices Stats:
{"num_choices": 23, "num_triton_choices": 23, "best_kernel": "triton_flex_attention_backward_10", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M1=64, BLOCK_M2=64, BLOCK_N1=64, BLOCK_N2=64, FLOAT32_PRECISION=\"'tf32'\", GQA_SHARED_HEADS=8, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=True, OUTPUT_LOGSUMEXP=True, OUTPUT_MAX=False, PRESCALE_QK=False, QK_HEAD_DIM=64, QK_HEAD_DIM_ROUNDED=64, ROWS_GUARANTEED_SAFE=False, SAFE_HEAD_DIM=True, SM_SCALE=0.125, SPARSE_KV_BLOCK_SIZE=128, SPARSE_Q_BLOCK_SIZE=128, V_HEAD_DIM=64, V_HEAD_DIM_ROUNDED=64, WRITE_DQ=True, num_stages=2, num_warps=4", "best_time": 0.5386239886283875, "best_triton_pos": 0}
Autotune Choices Stats:
{"num_choices": 23, "num_triton_choices": 23, "best_kernel": "triton_flex_attention_backward_34", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M1=64, BLOCK_M2=64, BLOCK_N1=64, BLOCK_N2=64, FLOAT32_PRECISION=\"'tf32'\", GQA_SHARED_HEADS=8, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=True, OUTPUT_LOGSUMEXP=Tr

Step,Training Loss
10,2.373500
20,2.399400
30,2.324000
40,2.270000
50,2.184200
60,2.059100
70,1.914300
80,1.831600
90,1.750500
100,1.691000


Autotune Choices Stats:
{"num_choices": 23, "num_triton_choices": 23, "best_kernel": "triton_flex_attention_backward_66", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M1=32, BLOCK_M2=32, BLOCK_N1=32, BLOCK_N2=32, FLOAT32_PRECISION=\"'tf32'\", GQA_SHARED_HEADS=8, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=False, OUTPUT_LOGSUMEXP=True, OUTPUT_MAX=False, PRESCALE_QK=False, QK_HEAD_DIM=64, QK_HEAD_DIM_ROUNDED=64, ROWS_GUARANTEED_SAFE=False, SAFE_HEAD_DIM=True, SM_SCALE=0.125, SPARSE_KV_BLOCK_SIZE=128, SPARSE_Q_BLOCK_SIZE=128, V_HEAD_DIM=64, V_HEAD_DIM_ROUNDED=64, WRITE_DQ=True, num_stages=2, num_warps=4", "best_time": 2.0879039764404297, "best_triton_pos": 0}
Autotune Choices Stats:
{"num_choices": 23, "num_triton_choices": 23, "best_kernel": "triton_flex_attention_backward_90", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M1=32, BLOCK_M2=32, BLOCK_N1=32, BLOCK_N2=32, FLOAT32_PRECISION=\"'tf32'\", GQA_SHARED_HEADS=8, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=False, OUTPUT_LOGSUMEXP=

TrainOutput(global_step=3330, training_loss=0.6756624730857643, metrics={'train_runtime': 39927.6788, 'train_samples_per_second': 5.302, 'train_steps_per_second': 0.083, 'total_flos': 3.591204046200373e+18, 'train_loss': 0.6756624730857643, 'epoch': 30.0})

### 推論テスト

In [3]:
!ls -R kakidash_ultimate_expert/checkpoint-3330/

kakidash_ultimate_expert/checkpoint-3330/:
README.md		   optimizer.pt		    tokenizer.json
adapter_config.json	   rng_state.pth	    tokenizer_config.json
adapter_model.safetensors  scheduler.pt		    trainer_state.json
chat_template.jinja	   special_tokens_map.json  training_args.bin


In [1]:
from unsloth import FastLanguageModel
import torch

# 1. 保存したディレクトリからロード
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "kakidash_ultimate_expert/checkpoint-3330", # 保存したディレクトリ名
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 2. 推論モードへ切り替え（ここでパッチが推論用に最適化されます）
FastLanguageModel.for_inference(model)

# 3. エラー回避のための重要なフラグ設定
# Unsloth独自の高速generateではなく、標準的なgenerateを強制します
model.config.use_cache = True


# 3. 質問の実行
question = "身体障害者手帳を使って、有料道路の割引を受けるための具体的な手続きと必要なものを教えてください。"
prompt = f"### 質問:\n{question}\n\n### 回答:\n"

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs, 
    max_new_tokens = 300,
    temperature = 0.1,
    repetition_penalty = 1.2,
)

print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.8: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260322. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [01:14<00:00, 18.74s/it]


Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
### 質問:
身体障害者手帳を使って、有料道路の割引を受けるための具体的な手続きと必要なものを教えてください。

### 回答:
各都市町村で交通休止証明書を発行申請し、交付を受けた後に、自分自身もしくは代理人が所定の窓口へ提出することで利用できます。必要なのは身体障害者手帳（第1号）と透明帽です。ドキュメントには「転出した場合は新居国や法人名を書いた通告書」が必要になるケースとして記載されています。


In [9]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

# 質問
question = "身体障害者手帳を使って、有料道路の割引を受けるための具体的な手続きと必要なものを教えてください。"
prompt = f"### 質問:\n{question}\n\n### 回答:\n"

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 丁寧に生成
outputs = model.generate(
    **inputs, 
    max_new_tokens = 300,
    temperature = 0.1,    # 学習した「正解」を忠実に出力させる
    repetition_penalty = 1.2,
    use_cache = False
)

print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

AcceleratorError: CUDA error: operation not permitted
Search for `cudaErrorNotPermitted' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
